# Walmart Market Analysis — historical draft

Recovered local coursework notebook. Outputs and execution counts were removed. Syntax-invalid planning cells are retained as markdown. Requires separately obtained source CSV files; end-to-end execution is not verified.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import numpy as np

In [ ]:
dowandjones = pd.read_csv("dowandjones.csv")

dowandjones.head()

In [ ]:
def rename_dataframe(dataframe):
    return dataframe.rename(columns = {'Date':'date',
                                            'Close/Last':'closing',
                                            'Open': 'open',
                                            'High': 'high',
                                            'Low': 'low'
                                   })

In [ ]:
dowandjones = dowandjones.drop(columns=['Adj Close**', 'Volume'])

dowandjones['Date'] = pd.to_datetime(dowandjones['Date'], format='%b %d, %Y')
dowandjones.rename(columns={'Close*': 'closing'}, inplace=True)

#dowandjones.rename(columns={'Date': 'date'}, inplace=True)

print(dowandjones.head())

In [ ]:
dowandjones = rename_dataframe(dowandjones)
dowandjones.head()

In [ ]:
nasdaq = pd.read_csv("nasdaq.csv")
nasdaq.head()

In [ ]:
walmart = pd.read_csv("walmart.csv")
walmart.head()

In [ ]:
sandp = pd.read_csv("sandp500.csv")
sandp.head()

In [ ]:
def convert_dates(dataframe):
    dataframe['date'] = pd.to_datetime(dataframe['date'])
    dataframe['month'] = dataframe['date'].dt.month
    dataframe['day'] = dataframe['date'].dt.day
    dataframe['year'] = dataframe['date'].dt.year
    return dataframe

In [ ]:
sandp = rename_dataframe(sandp)
sandp = convert_dates(sandp)
sandp.head()

In [ ]:
walmart = rename_dataframe(walmart)
walmart = convert_dates(walmart)
walmart.head()

In [ ]:
nasdaq = rename_dataframe(nasdaq)
nasdaq = convert_dates(nasdaq)
nasdaq.head()

In [ ]:
dowandjones = convert_dates(dowandjones)

In [ ]:
sandp = sandp[sandp['year'].isin(walmart['year'])]
dowandjones = dowandjones[dowandjones['year'].isin(sandp['year'])]
nasdaq = nasdaq[nasdaq['year'].isin(nasdaq['year'])]

In [ ]:
sandp['year'].unique()

In [ ]:
dowandjones['year'].unique()

In [ ]:
nasdaq['year'].unique()

In [ ]:
def standardize_column_case(dataframe):
    dataframe.columns = [col.title() for col in dataframe.columns]
    return dataframe

In [ ]:
dowandjones = standardize_column_case(dowandjones)
nasdaq = standardize_column_case(nasdaq)
walmart = standardize_column_case(walmart)
sandp = standardize_column_case(sandp)

In [ ]:
print(dowandjones.columns)
print(nasdaq.columns)
print(walmart.columns)
print(sandp.columns)


numbers and strings = help
conditionals = help
iteration = help
pandas = good!
functions = for calculating monthly average???
should we load the other datasets??
numpy = good!
visualization = good
linear regression = good

In [ ]:
merged_data = pd.merge(dowandjones, sandp, on='Date', how='inner', suffixes=('_dow', '_sp'))
merged_data = pd.merge(merged_data, walmart, on='Date', how='inner', suffixes=('', '_walmart'))
merged_data = pd.merge(merged_data, nasdaq, on='Date', how='inner', suffixes=('', '_nasdaq'))

In [ ]:
merged_data = merged_data.drop(columns=['Month', 'Day', 'Year', 'Month_dow', 'Day_dow', 'Year_dow', 
                                        'Month_sp', 'Day_sp', 'Year_sp', 'Month_nasdaq', 'Day_nasdaq', 'Year_nasdaq'])

In [ ]:
#merged_data.columns = merged_data.columns.str.lower()

In [ ]:
merged_data.head()

In [ ]:
for col in merged_data.columns:
    if merged_data[col].dtype == 'object': 
        merged_data[col] = merged_data[col].str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)


In [ ]:
print(merged_data.dtypes)

In [ ]:
merged_data.to_csv("merged_data.csv", index=False)

In [ ]:
print(merged_data[['High_nasdaq', 'Low_nasdaq']].head())
print(merged_data[['High_nasdaq', 'Low_nasdaq']].dtypes)

In [ ]:
merged_data = merged_data.loc[:, ~merged_data.columns.duplicated()]

In [ ]:

merged_data = merged_data.copy()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(merged_data['Date'], merged_data['Closing_dow'], label='Dow Jones', alpha=0.7)
plt.plot(merged_data['Date'], merged_data['Closing_sp'], label='S&P 500', alpha=0.7)
plt.plot(merged_data['Date'], merged_data['Closing_nasdaq'], label='Nasdaq', alpha=0.7)

plt.xlabel('Date')
plt.ylabel('Closing Price')
plt.title('Closing Prices: Major Indices')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(merged_data['Date'], merged_data['Closing'], label='Walmart', color='blue', linewidth=2)
plt.xlabel('Date')
plt.ylabel('Closing Price')
plt.title('Walmart Closing Prices')
plt.legend()
plt.show()

In [ ]:
merged_data['Walmart_Range'] = merged_data['High'] - merged_data['Low']
merged_data['Dow_Range'] = merged_data['High_dow'] - merged_data['Low_dow']
merged_data['Sp_Range'] = merged_data['High_sp'] - merged_data['Low_sp']
merged_data['Nasdaq_Range'] = merged_data['High_nasdaq'] - merged_data['Low_nasdaq']

merged_data['Walmart_Range_Smoothed'] = merged_data['Walmart_Range'].rolling(window=7).mean()
merged_data['Dow_Range_Smoothed'] = merged_data['Dow_Range'].rolling(window=7).mean()
merged_data['Sp_Range_Smoothed'] = merged_data['Sp_Range'].rolling(window=7).mean()
merged_data['Nasdaq_Range_Smoothed'] = merged_data['Nasdaq_Range'].rolling(window=7).mean()

plt.figure(figsize=(12, 6))
plt.plot(merged_data['Date'], merged_data['Dow_Range_Smoothed'], label='Dow Jones', alpha=0.7)
plt.plot(merged_data['Date'], merged_data['Sp_Range_Smoothed'], label='S&P 500', alpha=0.7)
plt.plot(merged_data['Date'], merged_data['Nasdaq_Range_Smoothed'], label='Nasdaq', alpha=0.7)

plt.xlabel('Date')
plt.ylabel('Price Range (High - Low)')
plt.title('Daily Price Range: Major Indices')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(merged_data['Date'], merged_data['Walmart_Range_Smoothed'], label='Walmart', linewidth=2, color='blue')

plt.xlabel('Date')
plt.ylabel('Price Range (High - Low)')
plt.title('Daily Price Range: Walmart')
plt.legend()
plt.show()

In [ ]:
merged_data['Walmart_Change'] = (merged_data['Closing'] - merged_data['Open']) / merged_data['Open']
merged_data['Dow_Change'] = (merged_data['Closing_dow'] - merged_data['Open_dow']) / merged_data['Open_dow']
merged_data['Sp_Change'] = (merged_data['Closing_sp'] - merged_data['Open_sp']) / merged_data['Open_sp']
merged_data['Nasdaq_Change'] = (merged_data['Closing_nasdaq'] - merged_data['Open_nasdaq']) / merged_data['Open_nasdaq']

merged_data['Dow_Change_Smoothed'] = merged_data['Dow_Change'].rolling(window=7).mean()
merged_data['Sp_Change_Smoothed'] = merged_data['Sp_Change'].rolling(window=7).mean()
merged_data['Nasdaq_Change_Smoothed'] = merged_data['Nasdaq_Change'].rolling(window=7).mean()
merged_data['Walmart_Change_Smoothed'] = merged_data['Walmart_Change'].rolling(window=7).mean()

plt.figure(figsize=(12, 6))
plt.plot(merged_data['Date'], merged_data['Dow_Change_Smoothed'], label='Dow Jones Weekly', linewidth = 2,alpha=0.5)
plt.plot(merged_data['Date'], merged_data['Sp_Change_Smoothed'], label='S&P 500, Weekly', linewidth = 2,alpha=0.5)
plt.plot(merged_data['Date'], merged_data['Nasdaq_Change_Smoothed'], label='Nasdaq, Weekly', linewidth = 2,alpha=0.5)

plt.xlabel('Date')
plt.ylabel('Daily Percentage Change')
plt.title('Daily Percentage Change:Major Indices')
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(merged_data['Date'], merged_data['Walmart_Change_Smoothed'], label='Walmart', linewidth=2, color='green')

plt.xlabel('Date')
plt.ylabel('Daily Percentage Change')
plt.title('Daily Percentage Change: Walmart')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(merged_data['Date'], merged_data['Dow_Change_Smoothed'], label='Dow Jones', alpha=0.7)

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(merged_data['Date'], merged_data['Sp_Change_Smoothed'], label='S&P 500', alpha=0.7)

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(merged_data['Date'], merged_data['Nasdaq_Change_Smoothed'], label='Nasdaq', alpha=0.7)

In [ ]:
volume_correlation = merged_data[['Volume', 'Closing_dow', 'Closing_sp', 'Closing_nasdaq']].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(volume_correlation, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Between Walmart Volume and Major Indices')
plt.show()

In [ ]:
post_covid = merged_data[merged_data['Date'] >= '2020-01-01'].copy()
pre_covid = merged_data[merged_data['Date'] < '2020-01-01'].copy()

post_covid['Walmart_Range'] = post_covid['High'] - post_covid['Low']
post_covid['Dow_Range'] = post_covid['High_dow'] - post_covid['Low_dow']
post_covid['Sp_Range'] = post_covid['High_sp'] - post_covid['Low_sp']
post_covid['Nasdaq_Range'] = post_covid['High_nasdaq'] - post_covid['Low_nasdaq']


In [ ]:
post_covid['Walmart_Range'] = post_covid['High'] - post_covid['Low'].rolling(window=7).mean()
post_covid['Dow_Range'] = post_covid['High_dow'] - post_covid['Low_dow'].rolling(window=7).mean()
post_covid['Sp_Range'] = post_covid['High_sp'] - post_covid['Low_sp'].rolling(window=7).mean()
post_covid['Nasdaq_Range'] = post_covid['High_nasdaq'] - post_covid['Low_nasdaq'].rolling(window=7).mean()


plt.figure(figsize=(12, 6))
plt.plot(post_covid['Date'], post_covid['Dow_Range'], label='Dow Jones (Range)', alpha=0.7)
plt.plot(post_covid['Date'], post_covid['Sp_Range'], label='S&P 500 (Range)', alpha=0.7)
plt.plot(post_covid['Date'], post_covid['Nasdaq_Range'], label='Nasdaq (Range)', alpha=0.7)

plt.xlabel('Date')
plt.ylabel('Price Range (High - Low)')
plt.title('Daily Price Range During Covid: Major Indices')
plt.legend()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(post_covid['Date'], post_covid['Walmart_Range'], label='Walmart (Range)', linewidth=2, color='orange')

plt.xlabel('Date')
plt.ylabel('Price Range (High - Low)')
plt.title('Daily Price Range During Covid: Walmart')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(pre_covid['Date'], pre_covid['Closing'], label='Walmart', linewidth=2, color='blue')

plt.xlabel('Date')
plt.ylabel('Closing Price')
plt.title('Walmart Closing Price (Pre-Covid)')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(post_covid['Date'], post_covid['Closing'], label='Walmart', linewidth=2, color='blue')

plt.xlabel('Date')
plt.ylabel('Closing Price')
plt.title('Walmart Closing Price (Post-Covid)')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(pre_covid['Date'], pre_covid['Closing_dow'], label='Dow Jones', linewidth=2, color='orange', alpha=0.8)
plt.plot(pre_covid['Date'], pre_covid['Closing_sp'], label='S&P 500', linewidth=2, color='green', alpha=0.8)
plt.plot(pre_covid['Date'], pre_covid['Closing_nasdaq'], label='Nasdaq', linewidth=2, color='red', alpha=0.8)

plt.xlabel('Date')
plt.ylabel('Closing Price')
plt.title('Major Indices Closing Prices (Pre-Covid)')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(post_covid['Date'], post_covid['Closing_dow'], label='Dow Jones', linewidth=2, color='orange', alpha=0.8)
plt.plot(post_covid['Date'], post_covid['Closing_sp'], label='S&P 500', linewidth=2, color='green', alpha=0.8)
plt.plot(post_covid['Date'], post_covid['Closing_nasdaq'], label='Nasdaq', linewidth=2, color='red', alpha=0.8)

plt.xlabel('Date')
plt.ylabel('Closing Price')
plt.title('Major Indices Closing Prices (Post-Covid)')
plt.legend()
plt.show()

In [ ]:
merged_data['Date'] = pd.to_datetime(merged_data['Date'])

merged_data['Month'] = merged_data['Date'].dt.to_period('M')
monthly_avg = merged_data.groupby('Month')['Closing'].mean()

plt.figure(figsize=(12, 6))
plt.plot(monthly_avg.index.astype(str), monthly_avg.values, marker='o', color='blue', label='Monthly Avg Price')
plt.xlabel('Month')
plt.ylabel('Average Closing Price')
plt.title('Walmart Monthly Average Closing Price')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
merged_data['Week'] = merged_data['Date'].dt.to_period('W')
weekly_ranges = merged_data.groupby('Week').apply(
    lambda x: x['High'] - x['Low']
)

plt.figure(figsize=(8, 10))
plt.boxplot(weekly_ranges, patch_artist=True, notch=True, showmeans=True)
plt.xlabel('Weeks')
plt.ylabel('Price Range (High - Low)')
plt.title('Weekly Price Volatility: Walmart')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.scatter(merged_data['Closing_dow'], merged_data['Closing'], alpha=0.7, c='green')

coefficients = np.polyfit(merged_data['Closing_dow'],merged_data['Closing'],1)
line = np.polyval(coefficients,merged_data['Closing_dow'])
plt.plot(merged_data['Closing_dow'],line,color = 'red',label = 'Best Fit Line')

plt.xlabel('Closing Price of Dow Jones')
plt.ylabel('Closing Price of Walmart')
plt.title('Walmart: Relationship Between Closing Price of Dow Jones and Walmart')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
#print(merged_data.columns)
#merged_data.head()

selected_columns = merged_data [['Closing_dow','Closing_sp','Closing','Closing_nasdaq']]
print(selected_columns)

In [ ]:
plt.figure(figsize=(12, 6))
plt.scatter(merged_data['Closing_sp'], merged_data['Closing'], alpha=0.7, c='orange')

coefficients = np.polyfit(merged_data['Closing_sp'],merged_data['Closing'],1)
line = np.polyval(coefficients,merged_data['Closing_sp'])
plt.plot(merged_data['Closing_sp'],line,color = 'red',label = 'Best Fit Line')


plt.xlabel('Closing price of S&P 500')
plt.ylabel('Closing Price of Walmart')
plt.title('Relationship between Closing Price of S&P 500 and Walmart')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.scatter(merged_data['Closing_nasdaq'], merged_data['Closing'], alpha=0.7, c='blue')

coefficients = np.polyfit(merged_data['Closing_nasdaq'],merged_data['Closing'],1)
line = np.polyval(coefficients,merged_data['Closing_nasdaq'])
plt.plot(merged_data['Closing_nasdaq'],line,color = 'red',label = 'Best Fit Line')

plt.xlabel('Closing price of S&P 500')
plt.ylabel('Closing Price of Walmart')
plt.title('Relationship between Closing Price of S&P 500 and Walmart')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
merged_data['Change'] = merged_data['Closing'] - merged_data['Open']
up_days = (merged_data['Change'] > 0).sum()
down_days = (merged_data['Change'] <= 0).sum()

plt.figure(figsize=(6, 6))
plt.pie([up_days, down_days], labels=['Up Days', 'Down Days'], autopct='%1.1f%%', colors=['blue', 'orange'], startangle=140)
plt.title('Walmart: Daily Price Movement (Up vs Down Days)')
plt.tight_layout()
plt.show()